# Baseline: TF-IDF + metadata (level-1A)

**Mục đích:** huấn luyện các mô hình classical trên bộ đặc trưng đã tạo ở `preprocessing.ipynb`, đánh giá bằng **accuracy trên dev**, và lưu xác suất để dùng cho stacking ở bước sau.

> **"Baseline" ở đây** = 4 mô hình classical trên TF-IDF + metadata. Không phải Text-CNN/LSTM/GRU của bài báo (những mô hình đó thuộc nhánh PhoBERT/deep learning ở bước sau).

**Quy ước đánh giá:**
- Chỉ số chính: **accuracy trên `dev`**.
- Macro-F1 và confusion matrix chỉ để tham khảo.
- Không dùng `dev` để fit bất kỳ thứ gì — `dev` là hold-out.

**Đầu vào:** `artifacts/sparse/X_{train,dev,test}.npz`, `artifacts/labels/y_*.npy`, `artifacts/models/preprocessors.joblib`.

## 1. Nạp artifacts & chuẩn bị ma trận

Khối **dense** nằm ở đầu ma trận; cần scale riêng vì các đặc trưng như `n_chars` (0–1089) sẽ lấn át TF-IDF nếu để nguyên. Dùng `StandardScaler(with_mean=False)` để giữ ma trận thưa.

In [ ]:
# ============================================================
# 1. IMPORT & LOAD ARTIFACTS
# ============================================================
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
SEED = 42


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PROB_DIR = ARTIFACT_DIR / "probs" / "no-oof"
PROB_DIR.mkdir(parents=True, exist_ok=True)

X_train = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_train.npz").tocsr()
X_dev = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_dev.npz").tocsr()
X_test = sparse.load_npz(ARTIFACT_DIR / "sparse" / "X_test.npz").tocsr()
y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
y_dev = np.load(ARTIFACT_DIR / "labels" / "y_dev.npy")
pre = joblib.load(ARTIFACT_DIR / "models" / "preprocessors.joblib")

N_DENSE = len(pre["dense_feature_names"])
print("X_train", X_train.shape, "| X_dev", X_dev.shape, "| X_test", X_test.shape)
print("dense features:", N_DENSE)
print("Nhãn train:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Nhãn dev  :", dict(zip(*np.unique(y_dev, return_counts=True))))
print("Majority baseline (dev):", round((y_dev == 0).mean(), 4))

In [ ]:
# ============================================================
# 1.2 SCALE DENSE + GHÉP LẠI
# ============================================================
scaler = StandardScaler(with_mean=False)
scaler.fit(X_train[:, :N_DENSE])


def assemble(X):
    """Scale riêng khối dense, giữ nguyên TF-IDF."""
    dense = scaler.transform(X[:, :N_DENSE])
    return sparse.hstack([dense, X[:, N_DENSE:]], format="csr").astype(np.float32)


A_train = assemble(X_train)
A_dev = assemble(X_dev)
A_test = assemble(X_test)

print("Sau scale:", A_train.shape, A_dev.shape, A_test.shape)
print("nnz train:", f"{A_train.nnz:,}", "| dtype:", A_train.dtype)

## 2. Hàm đánh giá dùng chung

Mỗi mô hình được fit trên `train`, dự đoán `dev`, in accuracy / macro-F1 / confusion matrix và lưu xác suất 4 lớp cho `dev` + `test`.

In [ ]:
# ============================================================
# 2. HÀM ĐÁNH GIÁ & LƯU XÁC SUẤT
# ============================================================
RESULTS = []


def evaluate(name, estimator, Xtr=None, Xdv=None, Xte=None, save=True):
    Xtr = A_train if Xtr is None else Xtr
    Xdv = A_dev if Xdv is None else Xdv
    Xte = A_test if Xte is None else Xte
    t0 = time.time()
    estimator.fit(Xtr, y_train)
    fit_time = time.time() - t0

    pred = estimator.predict(Xdv)
    acc = accuracy_score(y_dev, pred)
    macro_f1 = f1_score(y_dev, pred, average="macro")

    print("=" * 78)
    print(f"{name}  |  accuracy={acc:.4f}  |  macro-F1={macro_f1:.4f}  |  "
          f"fit={fit_time:.1f}s")
    print("=" * 78)
    print(classification_report(y_dev, pred, digits=3, zero_division=0))
    print("Phân bố dự đoán:", pd.Series(pred).value_counts().sort_index().to_dict())
    print("Confusion matrix (hàng=thật, cột=dự đoán):")
    print(pd.DataFrame(
        confusion_matrix(y_dev, pred, labels=[0, 1, 2, 3]),
        index=[f"true_{i}" for i in range(4)],
        columns=[f"pred_{i}" for i in range(4)],
    ).to_string())

    if save and hasattr(estimator, "predict_proba"):
        p_dev = estimator.predict_proba(Xdv).astype(np.float32)
        p_test = estimator.predict_proba(Xte).astype(np.float32)
        np.save(PROB_DIR / f"{name}_dev.npy", p_dev)
        np.save(PROB_DIR / f"{name}_test.npy", p_test)
        print(f"Đã lưu xác suất: {name}_dev.npy {p_dev.shape}, {name}_test.npy {p_test.shape}")
        print("Xác suất trung bình mỗi lớp (dev):",
              np.round(p_dev.mean(axis=0), 4).tolist())

    RESULTS.append({
        "model": name,
        "accuracy": round(float(acc), 5),
        "macro_f1": round(float(macro_f1), 5),
        "fit_seconds": round(float(fit_time), 1),
    })
    return estimator

## 3. Logistic Regression (multinomial)

Mô hình tuyến tính ổn định trên TF-IDF thưa, cho xác suất trực tiếp. Với sklearn ≥ 1.5, bài toán đa lớp tự dùng multinomial.

In [ ]:
# ============================================================
# 3. LOGISTIC REGRESSION
# ============================================================
lr = evaluate(
    "logreg",
    LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=SEED),
)

## 4. LinearSVC + calibration

`LinearSVC` thường rất mạnh trên text nhiều chiều nhưng không có `predict_proba`, nên bọc trong `CalibratedClassifierCV` (sigmoid, 3 fold) để lấy xác suất cho stacking.

In [ ]:
# ============================================================
# 4. LINEARSVC + CALIBRATION
# ============================================================
svc = evaluate(
    "linsvc",
    CalibratedClassifierCV(
        estimator=LinearSVC(C=0.5, max_iter=5000, random_state=SEED),
        cv=3,
        method="sigmoid",
    ),
)

## 5. Complement Naive Bayes

ComplementNB yêu cầu đầu vào **không âm**, trong khi khối dense có `service_minus_product` mang giá trị âm. Vì vậy NB được train trên **TF-IDF thuần** (đúng chuẩn cho NB) — vẫn hữu ích như một "giọng nói" độc lập trong stacking.

In [ ]:
# ============================================================
# 5. COMPLEMENT NB (trên TF-IDF thuần, không âm)
# ============================================================
nb = evaluate(
    "complement_nb",
    ComplementNB(alpha=1.0),
    Xtr=X_train[:, N_DENSE:],
    Xdv=X_dev[:, N_DENSE:],
    Xte=X_test[:, N_DENSE:],
)

## 6. SGDClassifier (modified Huber)

Thay thế nhẹ cho LightGBM khi chưa cài `lightgbm`; cho xác suất và chạy rất nhanh. Nếu đã cài LightGBM, có thể thêm sau.

In [ ]:
# ============================================================
# 6. SGD CLASSIFIER
# ============================================================
sgd = evaluate(
    "sgd",
    SGDClassifier(
        loss="modified_huber", alpha=1e-5, max_iter=3000, tol=1e-4,
        random_state=SEED,
    ),
)

In [ ]:
# ============================================================
# 6.2 LIGHTGBM (tuỳ chọn)
# ============================================================
try:
    from lightgbm import LGBMClassifier

    lgbm = evaluate(
        "lightgbm",
        LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8, random_state=SEED,
            n_jobs=-1, verbose=-1,
        ),
    )
except ImportError:
    print("LightGBM chưa cài — bỏ qua bước này (không ảnh hưởng kết quả baseline).")

## 7. Tổng hợp & so sánh với TF-IDF thuần

Câu hỏi cần trả lời: **metadata có thực sự giúp không?** Ta train thêm một `LinearSVC` chỉ trên TF-IDF (bỏ 52 đặc trưng dense) để so sánh trực tiếp.

In [ ]:
# ============================================================
# 7. SO SÁNH: TF-IDF + METADATA vs TF-IDF THUẦN
# ============================================================
print("Đang train LinearSVC chỉ trên TF-IDF (bỏ metadata)...")
t0 = time.time()
svc_tfidf_only = CalibratedClassifierCV(
    estimator=LinearSVC(C=0.5, max_iter=5000, random_state=SEED),
    cv=3, method="sigmoid",
)
svc_tfidf_only.fit(X_train[:, N_DENSE:], y_train)
pred_only = svc_tfidf_only.predict(X_dev[:, N_DENSE:])
acc_only = accuracy_score(y_dev, pred_only)
print(f"TF-IDF thuần: accuracy={acc_only:.4f}  ({time.time() - t0:.1f}s)")

best_full = max(RESULTS, key=lambda r: r["accuracy"])
print(f"Tốt nhất có metadata: {best_full['model']} = {best_full['accuracy']:.4f}")
print(f"Chênh lệch metadata: {best_full['accuracy'] - acc_only:+.4f}")

RESULTS.append({
    "model": "linsvc_tfidf_only",
    "accuracy": round(float(acc_only), 5),
    "macro_f1": None,
    "fit_seconds": round(float(time.time() - t0), 1),
})

In [ ]:
# ============================================================
# 7.2 BẢNG TỔNG HỢP & LƯU KẾT QUẢ
# ============================================================
summary = pd.DataFrame(RESULTS).sort_values("accuracy", ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

best = summary.iloc[0]
print(f"\nMÔ HÌNH TỐT NHẤT: {best['model']} — accuracy {best['accuracy']:.4f}")

print("\nXác suất đã lưu trong:", PROB_DIR)
for p in sorted(PROB_DIR.glob("*.npy")):
    print(f"  {p.name:28s} {p.stat().st_size / 1e6:.2f} MB")

## 8. Lưu model phân loại

Lưu các estimator đã fit vào `artifacts/models/classifiers/` để có thể tái sử dụng mà không phải train lại. Cell dưới đây cũng **nạp lại từng model** và kiểm tra accuracy trên dev để chắc file lưu không bị lỗi.

In [ ]:
# ============================================================
# 8. LƯU MODEL PHÂN LOẠI
# ============================================================
MODELS_DIR = ARTIFACT_DIR / "models" / "classifiers"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

fitted_models = {
    "logreg": lr,
    "linsvc": svc,
    "complement_nb": nb,
    "sgd": sgd,
}
if "lgbm" in globals():
    fitted_models["lightgbm"] = lgbm

saved_paths = {}
for name, model in fitted_models.items():
    path = MODELS_DIR / f"{name}.joblib"
    joblib.dump(model, path, compress=3)
    saved_paths[name] = path
    print(f"Đã lưu {name:14s} → {path.relative_to(PROJECT_ROOT)}  "
          f"({path.stat().st_size / 1e6:.2f} MB)")

print("\nKiểm tra nạp lại và accuracy trên dev:")
for name, path in saved_paths.items():
    reloaded = joblib.load(path)
    X_check = X_dev[:, N_DENSE:] if name == "complement_nb" else A_dev
    pred = reloaded.predict(X_check)
    acc = accuracy_score(y_dev, pred)
    print(f"  {name:14s} reload OK | dev accuracy = {acc:.4f}")

print("\nTổng dung lượng model:",
      f"{sum(p.stat().st_size for p in saved_paths.values()) / 1e6:.2f} MB")

## 9. Nhận xét & bước tiếp theo

**Điều cần kiểm tra:**

1. Mô hình tốt nhất có vượt **~88.5%** không? Nếu không, quay lại kiểm tra scale khối dense và cấu hình TF-IDF.
2. Metadata có giúp không? Nếu chênh lệch ≤ 0, có thể bỏ bớt đặc trưng nhiễu.
3. Phân bố dự đoán có sụp về một lớp không? Nếu có, xem lại class weight / dữ liệu.

**Bước tiếp theo (hướng C):**

1. **OOF 5-fold** cho các model này để có xác suất sạch cho stacking.
2. Cài PyTorch CUDA + transformers (song song).
3. Tách từ cho PhoBERT (`comment_seg`), fine-tune fold 0.
4. Stacking: nối xác suất OOF của tất cả base models + metadata → meta-learner.
5. Sinh `submission.csv`.

**Lưu ý:** xác suất `test` lưu ở đây được sinh từ model fit trên **toàn bộ train** — dùng làm phương án dự phòng. Khi làm stacking, nên dùng trung bình xác suất của các model fold để ổn định hơn.

**Model đã lưu:** `artifacts/models/classifiers/{logreg,linsvc,complement_nb,sgd,lightgbm}.joblib`. Nạp lại bằng:

```python
import joblib
model = joblib.load("artifacts/models/classifiers/lightgbm.joblib")
pred = model.predict(X_new)
```